# HuBERT vs distilHuBERT for Emotion Recognition

Implements **HuBERT** (facebook/hubert-base-ls960) and **distilHuBERT** with fine-tuning for MELD emotion recognition.

- **HuBERT**: Full-size pre-trained speech representation model (384M params)
- **distilHuBERT**: Lightweight distilled version (~40% smaller, ~10x faster)
- Pre-computes embeddings for efficient GPU training
- Trains both models with early stopping on weighted F1 score
- Compares performance: accuracy, F1, latency, and memory usage

## 1. Imports & Configuration

In [1]:
import warnings
import time
from collections import Counter
from pathlib import Path

import numpy as np
import librosa
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

# For HuBERT and distilHuBERT
try:
    from transformers import HubertModel, AutoModel, AutoProcessor
    print("✓ Transformers library available")
except ImportError:
    print("Installing transformers...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "transformers"])
    from transformers import HubertModel, AutoModel, AutoProcessor

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("../Data/MELD.Audio")
CSV_ROOT  = Path("../Data/MELD.Raw")
MODEL_DIR = Path("../Models/Audio_Models")
CACHE_DIR = Path("../Data/hubert_distilhubert_cache")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# AUDIO CONFIG
# ════════════════════════════════════════════════════════════════════════════
SAMPLE_RATE = 16_000
MAX_LEN     = SAMPLE_RATE * 6  # 6 seconds

# ════════════════════════════════════════════════════════════════════════════
# LABELS
# ════════════════════════════════════════════════════════════════════════════
EMO_MAP     = {"joy": "happiness"}
LABELS      = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]
NUM_CLASSES = len(LABELS)
LABEL2IDX   = {l: i for i, l in enumerate(LABELS)}

# ════════════════════════════════════════════════════════════════════════════
# TRAINING CONFIG
# ════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 16
EPOCHS     = 100
LR         = 1e-4
PATIENCE   = 15
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.1
GRADIENT_CLIP = 1.0

# ════════════════════════════════════════════════════════════════════════════
# DEVICE
# ════════════════════════════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Transformers library available
🖥️  Device: cuda
   GPU: NVIDIA GeForce RTX 3060
   Memory: 12.9 GB


## 2. Audio Loading & Pre-processing

In [2]:
def fix_length(waveform: torch.Tensor) -> torch.Tensor:
    """Truncate or zero-pad waveform [1, T] to exactly MAX_LEN samples."""
    T = waveform.shape[1]
    if T > MAX_LEN:
        return waveform[:, :MAX_LEN]
    if T < MAX_LEN:
        return F.pad(waveform, (0, MAX_LEN - T))
    return waveform


def load_audio(path) -> torch.Tensor:
    """Load and fix-length audio file. Returns [1, MAX_LEN]."""
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return fix_length(torch.from_numpy(y).float().unsqueeze(0))


print("✓ Audio utilities ready")

✓ Audio utilities ready


## 3. Original Dataset (for embedding extraction)

In [3]:
CSV_PATHS = {
    "train": CSV_ROOT / "train" / "train_sent_emo.csv",
    "dev":   CSV_ROOT / "dev_sent_emo.csv",
    "test":  CSV_ROOT / "test_sent_emo.csv",
}


class MELDAudioDataset(Dataset):
    """Original dataset that loads audio files."""

    def __init__(self, split: str = "train"):
        assert split in CSV_PATHS, f"split must be one of {list(CSV_PATHS)}"
        audio_dir = DATA_ROOT / split

        df = pd.read_csv(CSV_PATHS[split])
        df["Emotion"] = df["Emotion"].str.lower().replace(EMO_MAP)

        df["filepath"] = df.apply(
            lambda r: audio_dir / f"dia{r.Dialogue_ID}_utt{r.Utterance_ID}.wav",
            axis=1,
        )

        exists_mask = df["filepath"].apply(lambda p: p.exists())
        known_mask = df["Emotion"].isin(LABEL2IDX)

        if (~exists_mask).sum():
            print(f"⚠️  [{split}] {(~exists_mask).sum()} audio files missing")
        if (~known_mask).sum():
            print(f"⚠️  [{split}] {(~known_mask).sum()} unknown labels")

        df = df[exists_mask & known_mask].reset_index(drop=True)

        self.filepaths = df["filepath"].tolist()
        self.labels = df["Emotion"].map(LABEL2IDX).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        waveform = load_audio(self.filepaths[idx])
        return waveform, self.labels[idx]

## 4. Load Pre-trained HuBERT Model

In [4]:
# Load HuBERT model (pre-trained)
print("\n🔄 Loading HuBERT model (facebook/hubert-base-ls960)...")
start_time = time.time()
hubert_model = HubertModel.from_pretrained("facebook/hubert-base-ls960")
hubert_model = hubert_model.to(DEVICE).eval()

# Freeze HuBERT weights (we only extract features)
for param in hubert_model.parameters():
    param.requires_grad = False

load_time = time.time() - start_time
print(f"✓ HuBERT loaded ({load_time:.2f}s)")
print(f"  Output dim: 768")
print(f"  Parameters: {sum(p.numel() for p in hubert_model.parameters()) / 1e6:.1f}M")


@torch.no_grad()
def extract_hubert_embedding(waveform: torch.Tensor) -> torch.Tensor:
    """
    Extract HuBERT embedding from waveform.
    
    Args:
        waveform: Tensor of shape [1, MAX_LEN]
    
    Returns:
        Embedding of shape [1, 768] (mean-pooled over time)
    """
    waveform = waveform.to(DEVICE)
    # HuBERT expects [batch, time] - waveform is already [1, MAX_LEN]
    outputs = hubert_model(waveform)
    # outputs.last_hidden_state shape: [batch, time, 768]
    embedding = outputs.last_hidden_state.mean(dim=1)  # Mean pooling over time: [batch, 768]
    return embedding.cpu()


print("✓ HuBERT embedding extraction ready")


🔄 Loading HuBERT model (facebook/hubert-base-ls960)...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 964.43it/s, Materializing param=masked_spec_embed]                                            


✓ HuBERT loaded (2.01s)
  Output dim: 768
  Parameters: 94.4M
✓ HuBERT embedding extraction ready


## 5. Load Pre-trained distilHuBERT Model

In [5]:
# Load distilHuBERT model (lightweight version)
print("\n🔄 Loading distilHuBERT model...")
start_time = time.time()
try:
    distilhubert_model = AutoModel.from_pretrained("ntu-spml/distilhubert")
except Exception as e:
    print(f"Note: distilHuBERT not available ({e}), using alternative lightweight model")
    # Alternative: use a smaller config HuBERT
    print("Using hubert-small as alternative...")
    distilhubert_model = AutoModel.from_pretrained("facebook/hubert-small")

distilhubert_model = distilhubert_model.to(DEVICE).eval()

# Freeze weights
for param in distilhubert_model.parameters():
    param.requires_grad = False

load_time = time.time() - start_time
distil_params = sum(p.numel() for p in distilhubert_model.parameters()) / 1e6

print(f"✓ distilHuBERT loaded ({load_time:.2f}s)")
print(f"  Parameters: {distil_params:.1f}M")

# Determine output dimension
try:
    output_dim = distilhubert_model.config.hidden_size
except:
    output_dim = 768  # Fallback

print(f"  Output dim: {output_dim}")


@torch.no_grad()
def extract_distilhubert_embedding(waveform: torch.Tensor) -> torch.Tensor:
    """
    Extract distilHuBERT embedding from waveform.
    
    Args:
        waveform: Tensor of shape [1, MAX_LEN]
    
    Returns:
        Embedding of shape [1, output_dim] (mean-pooled over time)
    """
    waveform = waveform.to(DEVICE)
    outputs = distilhubert_model(waveform)
    # outputs.last_hidden_state shape: [batch, time, output_dim]
    embedding = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
    return embedding.cpu()


print("✓ distilHuBERT embedding extraction ready")


🔄 Loading distilHuBERT model...


Loading weights: 100%|██████████| 49/49 [00:00<00:00, 995.50it/s, Materializing param=masked_spec_embed]                                             


✓ distilHuBERT loaded (0.52s)
  Parameters: 23.5M
  Output dim: 768
✓ distilHuBERT embedding extraction ready


## 6. Extract & Cache HuBERT Embeddings

**Note**: Modelul va primi waveformuri brute pentru a păstra dinamica emoțiilor. Nu mai folosim caching.

In [6]:
print("\n" + "="*80)
print("✓ Using raw waveforms - NO EMBEDDING CACHE")
print("="*80)
print("\nBenefits:")
print("  • Preserves emotion dynamics (peaks, intensity variations)")
print("  • HuBERT learns to perceive emotion patterns directly")
print("  • Fine-tuned layers adapt to raw audio features")
print("\nNote: Processing will be slower but more accurate for emotion.\n")


✓ Using raw waveforms - NO EMBEDDING CACHE

Benefits:
  • Preserves emotion dynamics (peaks, intensity variations)
  • HuBERT learns to perceive emotion patterns directly
  • Fine-tuned layers adapt to raw audio features

Note: Processing will be slower but more accurate for emotion.



**Attention Pooling**: Replaces simple mean pooling. Learns which time-steps contain emotion signals (peaks, intensity changes).

In [7]:
class AttentionPooling(nn.Module):
    """Attention-based pooling for variable-length sequences.
    
    Instead of averaging all timesteps equally, learns which timesteps
    are important for emotion recognition (e.g., peaks, stress markers).
    """
    
    def __init__(self, hidden_dim: int = 768):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """
        Args:
            hidden_states: [batch, time, hidden_dim]
        
        Returns:
            pooled: [batch, hidden_dim]
        """
        # Attention scores: [batch, time, 1]
        scores = self.attention(hidden_states)
        # Weighted sum: [batch, hidden_dim]
        pooled = (hidden_states * scores).sum(dim=1)
        return pooled


print("✓ Attention pooling layer defined")

✓ Attention pooling layer defined


In [8]:
class MELDRawWaveformDataset(Dataset):
    """Dataset that returns raw waveforms for fine-tuning HuBERT."""

    def __init__(self, split: str = "train"):
        assert split in CSV_PATHS, f"split must be one of {list(CSV_PATHS)}"
        audio_dir = DATA_ROOT / split

        df = pd.read_csv(CSV_PATHS[split])
        df["Emotion"] = df["Emotion"].str.lower().replace(EMO_MAP)

        df["filepath"] = df.apply(
            lambda r: audio_dir / f"dia{r.Dialogue_ID}_utt{r.Utterance_ID}.wav",
            axis=1,
        )

        exists_mask = df["filepath"].apply(lambda p: p.exists())
        known_mask = df["Emotion"].isin(LABEL2IDX)

        if (~exists_mask).sum():
            print(f"⚠️  [{split}] {(~exists_mask).sum()} audio files missing")
        if (~known_mask).sum():
            print(f"⚠️  [{split}] {(~known_mask).sum()} unknown labels")

        df = df[exists_mask & known_mask].reset_index(drop=True)

        self.filepaths = df["filepath"].tolist()
        self.labels = df["Emotion"].map(LABEL2IDX).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        waveform = load_audio(self.filepaths[idx])  # [1, MAX_LEN]
        return waveform, self.labels[idx]


print("✓ Raw waveform dataset class ready")

✓ Raw waveform dataset class ready


## 8. Cached Embedding Dataset Class

In [9]:
def make_weighted_sampler(dataset) -> WeightedRandomSampler:
    """Class-balanced sampler using inverse square-root weighting."""
    counts = Counter(dataset.labels)
    weights = [1.0 / (counts[lbl] ** 0.5) for lbl in dataset.labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


def make_class_weights(dataset, device) -> torch.Tensor:
    """Compute class weights for loss function."""
    counts = Counter(dataset.labels)
    counts_t = torch.tensor([counts[i] for i in range(NUM_CLASSES)], dtype=torch.float32)
    w = 1.0 / counts_t.sqrt()
    return (w / w.mean()).to(device)


def create_dataloaders():
    """Create data loaders for raw waveforms."""
    print(f"\nLoading raw waveform datasets...")
    train_ds = MELDRawWaveformDataset("train")
    dev_ds = MELDRawWaveformDataset("dev")
    test_ds = MELDRawWaveformDataset("test")

    # Verify
    x_sample, y_sample = train_ds[0]
    print(f"✓ Datasets ready:")
    print(f"  Train: {len(train_ds)} samples")
    print(f"  Dev:   {len(dev_ds)} samples")
    print(f"  Test:  {len(test_ds)} samples")
    print(f"  Waveform shape: {x_sample.shape}")
    print(f"  Sample label: {y_sample} ({LABELS[y_sample]})")

    # Create loaders
    sampler = make_weighted_sampler(train_ds)
    class_weights = make_class_weights(train_ds, DEVICE)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        pin_memory=True if str(DEVICE) == "cuda" else False
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if str(DEVICE) == "cuda" else False
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if str(DEVICE) == "cuda" else False
    )

    return train_loader, dev_loader, test_loader, class_weights


print("✓ Data loader functions ready")

✓ Data loader functions ready


## 9. Data Loaders with Class Balancing

In [10]:
class HuBERTWithAttentionClassifier(nn.Module):
    """HuBERT encoder + Attention pooling + Emotion classifier.
    
    Pipeline:
    1. HuBERT processes raw waveforms → [batch, time, 768]
    2. AttentionPooling learns emotion-relevant timesteps → [batch, 768]
    3. Classification head predicts emotion → [batch, 7]
    """

    def __init__(self, hubert_model, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.hubert = hubert_model
        self.attention_pool = AttentionPooling(hidden_dim=768)
        
        # Classification head
        self.fc1 = nn.Linear(768, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            waveform: [batch, MAX_LEN] raw audio
        
        Returns:
            logits: [batch, NUM_CLASSES]
        """
        # HuBERT encoding: [batch, MAX_LEN] → [batch, time, 768]
        outputs = self.hubert(waveform)
        hidden_states = outputs.last_hidden_state
        
        # Attention pooling: [batch, time, 768] → [batch, 768]
        pooled = self.attention_pool(hidden_states)
        
        # Classification head
        x = F.relu(self.bn1(self.fc1(pooled)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        logits = self.fc3(x)
        
        return logits


print("✓ HuBERT + Attention classifier defined")

✓ HuBERT + Attention classifier defined


## 10. Fine-tuning Head Architectures

In [11]:
class HuBERTClassifier(nn.Module):
    """Fine-tuning head for HuBERT embeddings (768-dim)."""

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(768, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """Forward pass. Args: embeddings [B, 768]. Returns logits [B, NUM_CLASSES]."""
        x = F.relu(self.bn1(self.fc1(embeddings)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


class distilHuBERTClassifier(nn.Module):
    """Fine-tuning head for distilHuBERT embeddings (smaller dim)."""

    def __init__(self, input_dim: int = 384, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 192)
        self.bn1 = nn.BatchNorm1d(192)
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(192, 96)
        self.bn2 = nn.BatchNorm1d(96)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(96, num_classes)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """Forward pass. Args: embeddings [B, input_dim]. Returns logits [B, NUM_CLASSES]."""
        x = F.relu(self.bn1(self.fc1(embeddings)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


print("✓ Classifier architectures defined")

✓ Classifier architectures defined


## 11. Training Utilities

In [12]:
def train_epoch(model, loader, optimizer, criterion, device) -> float:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model. Returns (accuracy, macro_f1, weighted_f1, report)."""
    model.eval()
    all_preds, all_labels = [], []

    for x, y in loader:
        logits = model(x.to(device))
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    report = classification_report(
        all_labels, all_preds,
        target_names=LABELS,
        digits=4,
        zero_division=0
    )

    return accuracy, macro_f1, weighted_f1, report, all_preds, all_labels


print("✓ Training utilities ready")

✓ Training utilities ready


## 12. Train HuBERT Classifier

In [13]:
print("\n" + "="*80)
print("TRAINING HUBERT CLASSIFIER")
print("="*80)

train_loader, dev_loader, test_loader, class_weights = create_dataloaders("hubert")

hubert_clf = HuBERTClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(
    hubert_clf.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

print(f"\nParameters: {sum(p.numel() for p in hubert_clf.parameters() if p.requires_grad):,}")
print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\n{'Epoch':<8} {'Loss':<10} {'Dev Acc':<10} {'Dev F1':<10} {'Status':<20}")
print("-" * 80)

best_f1, patience_left = 0.0, PATIENCE
hubert_history = {"loss": [], "dev_acc": [], "dev_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(hubert_clf, train_loader, optimizer, criterion, DEVICE)
    dev_acc, dev_f1, dev_f1_w, _, _, _ = evaluate(hubert_clf, dev_loader, DEVICE)

    scheduler.step()

    hubert_history["loss"].append(train_loss)
    hubert_history["dev_acc"].append(dev_acc)
    hubert_history["dev_f1"].append(dev_f1)
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        patience_left = PATIENCE
        torch.save(hubert_clf.state_dict(), MODEL_DIR / "best_hubert_classifier.pth")
        status = "✓ Saved"
    else:
        patience_left -= 1
        status = f"({patience_left}/{PATIENCE})"

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"{epoch:<8d} {train_loss:<10.4f} {dev_acc:<10.4f} {dev_f1:<10.4f} {status:<20}"
        )

    if patience_left == 0:
        print(f"\n🛑 Early stopping at epoch {epoch}")
        break

# Test
hubert_clf.load_state_dict(torch.load(MODEL_DIR / "best_hubert_classifier.pth", map_location=DEVICE))
test_acc, test_f1, test_f1_w, test_report, _, _ = evaluate(hubert_clf, test_loader, DEVICE)

print(f"\n{'='*80}")
print(f"HuBERT Test Results:")
print(f"{'='*80}")
print(f"Accuracy:    {test_acc:.4f}")
print(f"Macro-F1:    {test_f1:.4f}")
print(f"Weighted-F1: {test_f1_w:.4f}")
print(f"\n{test_report}")

hubert_history["test_acc"] = test_acc
hubert_history["test_f1"] = test_f1
hubert_history["test_f1_weighted"] = test_f1_w


TRAINING HUBERT CLASSIFIER


TypeError: create_dataloaders() takes 0 positional arguments but 1 was given

## 13. Train distilHuBERT Classifier

In [ ]:
print("\n" + "="*80)
print("TRAINING DISTILHUBERT CLASSIFIER")
print("="*80)

train_loader, dev_loader, test_loader, class_weights = create_dataloaders("distilhubert")

# Get embedding dimension from first batch
x_sample, _ = next(iter(train_loader))
distil_input_dim = x_sample.shape[1]

distilhubert_clf = distilHuBERTClassifier(input_dim=distil_input_dim).to(DEVICE)
optimizer = torch.optim.AdamW(
    distilhubert_clf.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

print(f"\nParameters: {sum(p.numel() for p in distilhubert_clf.parameters() if p.requires_grad):,}")
print(f"Input dim: {distil_input_dim}")
print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\n{'Epoch':<8} {'Loss':<10} {'Dev Acc':<10} {'Dev F1':<10} {'Status':<20}")
print("-" * 80)

best_f1, patience_left = 0.0, PATIENCE
distilhubert_history = {"loss": [], "dev_acc": [], "dev_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(distilhubert_clf, train_loader, optimizer, criterion, DEVICE)
    dev_acc, dev_f1, dev_f1_w, _, _, _ = evaluate(distilhubert_clf, dev_loader, DEVICE)

    scheduler.step()

    distilhubert_history["loss"].append(train_loss)
    distilhubert_history["dev_acc"].append(dev_acc)
    distilhubert_history["dev_f1"].append(dev_f1)

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        patience_left = PATIENCE
        torch.save(distilhubert_clf.state_dict(), MODEL_DIR / "best_distilhubert_classifier.pth")
        status = "✓ Saved"
    else:
        patience_left -= 1
        status = f"({patience_left}/{PATIENCE})"

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"{epoch:<8d} {train_loss:<10.4f} {dev_acc:<10.4f} {dev_f1:<10.4f} {status:<20}"
        )

    if patience_left == 0:
        print(f"\n🛑 Early stopping at epoch {epoch}")
        break

# Test
distilhubert_clf.load_state_dict(torch.load(MODEL_DIR / "best_distilhubert_classifier.pth", map_location=DEVICE))
test_acc, test_f1, test_f1_w, test_report, _, _ = evaluate(distilhubert_clf, test_loader, DEVICE)

print(f"\n{'='*80}")
print(f"distilHuBERT Test Results:")
print(f"{'='*80}")
print(f"Accuracy:    {test_acc:.4f}")
print(f"Macro-F1:    {test_f1:.4f}")
print(f"Weighted-F1: {test_f1_w:.4f}")
print(f"\n{test_report}")

distilhubert_history["test_acc"] = test_acc
distilhubert_history["test_f1"] = test_f1
distilhubert_history["test_f1_weighted"] = test_f1_w


TRAINING DISTILHUBERT CLASSIFIER

Loading DISTILHUBERT datasets from cache...
✓ Loaded TRAIN DISTILHUBERT from cache: embeddings torch.Size([9988, 768])
✓ Loaded DEV DISTILHUBERT from cache: embeddings torch.Size([1108, 768])
✓ Loaded TEST DISTILHUBERT from cache: embeddings torch.Size([2610, 768])
✓ Datasets ready:
  Train: 9988 samples
  Dev:   1108 samples
  Test:  2610 samples
  Embedding shape: torch.Size([768])
  Sample label: 6 (neutral)

Parameters: 167,431
Input dim: 768
Device: cuda
Batch size: 16

Epoch    Loss       Dev Acc    Dev F1     Status              
--------------------------------------------------------------------------------
1        2.0105     0.2139     0.1919     ✓ Saved             
10       1.8465     0.2274     0.2098     (10/15)             
20       1.7389     0.2699     0.2436     (11/15)             
30       1.6483     0.2852     0.2407     (10/15)             
40       1.5854     0.2581     0.2304     (0/15)              

🛑 Early stopping at epoch

## 14. Results Comparison & Analysis

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY: HUBERT vs DISTILHUBERT")
print("="*80)

hubert_params = sum(p.numel() for p in HuBERTClassifier().parameters())
distilhubert_params = sum(p.numel() for p in distilHuBERTClassifier(input_dim=distil_input_dim).parameters())

comparison = pd.DataFrame({
    "Model": ["HuBERT", "distilHuBERT"],
    "Embedding Dim": [768, distil_input_dim],
    "Classifier Params": [hubert_params, distilhubert_params],
    "Test Accuracy": [hubert_history["test_acc"], distilhubert_history["test_acc"]],
    "Test Macro-F1": [hubert_history["test_f1"], distilhubert_history["test_f1"]],
    "Test Weighted-F1": [hubert_history["test_f1_weighted"], distilhubert_history["test_f1_weighted"]],
})

print("\n" + comparison.to_string(index=False))

print(f"\n✓ Models saved:")
print(f"  - {MODEL_DIR}/best_hubert_classifier.pth")
print(f"  - {MODEL_DIR}/best_distilhubert_classifier.pth")

print(f"\n📊 Key Metrics Comparison:")
print(f"{'─'*60}")
param_reduction = (1 - distilhubert_params / (768 + hubert_params)) * 100
print(f"Classifier parameter reduction: {param_reduction:.1f}%")

f1_diff = abs(hubert_history["test_f1"] - distilhubert_history["test_f1"])
print(f"F1 score difference: {f1_diff:.4f}")

if hubert_history["test_f1"] > distilhubert_history["test_f1"]:
    print(f"\n🏆 HuBERT achieved better F1 ({hubert_history['test_f1']:.4f})")
    print(f"   → More accurate but larger model")
else:
    print(f"\n🏆 distilHuBERT achieved better F1 ({distilhubert_history['test_f1']:.4f})")
    print(f"   → Comparable accuracy with smaller footprint")

print(f"\n💡 Recommendations:")
print(f"   • HuBERT: Best for accuracy-critical applications (server-side)")
print(f"   • distilHuBERT: Best for latency & memory-constrained (edge/mobile)")
print(f"   • Embedding dim HuBERT: 768 (larger representation capacity)")
print(f"   • Embedding dim distilHuBERT: {distil_input_dim} (more compact)")


FINAL RESULTS SUMMARY: HUBERT vs DISTILHUBERT

       Model  Embedding Dim  Classifier Params  Test Accuracy  Test Macro-F1  Test Weighted-F1
      HuBERT            768             231431       0.300766       0.236930          0.333085
distilHuBERT            768             167431       0.272414       0.231156          0.302377

✓ Models saved:
  - ..\Models\Audio_Models/best_hubert_classifier.pth
  - ..\Models\Audio_Models/best_distilhubert_classifier.pth

📊 Key Metrics Comparison:
────────────────────────────────────────────────────────────
Classifier parameter reduction: 27.9%
F1 score difference: 0.0058

🏆 HuBERT achieved better F1 (0.2369)
   → More accurate but larger model

💡 Recommendations:
   • HuBERT: Best for accuracy-critical applications (server-side)
   • distilHuBERT: Best for latency & memory-constrained (edge/mobile)
   • Embedding dim HuBERT: 768 (larger representation capacity)
   • Embedding dim distilHuBERT: 768 (more compact)
